In [1]:
import pandas as pd
import json
import random

df = pd.read_csv("../data/suicide_watch_sample_50k_v1.csv")

df.head()

,Unnamed: 0,selftext,created_utc,title,subreddit,subreddit_id,retrieved_on,post_id
0,41644,I want to die I have been depressed most of my...,1724306507,I wish I had the courage to do it,SuicideWatch,t5_2qpzs,1724306521,t3_1eyc0m5
1,94933,"My brain is always going at 1000mph, thinking ...",1734772989,Brain is always going 1000mph,SuicideWatch,t5_2qpzs,1734773002,t3_1hj6cy8
2,68635,Bro-I just took like 4 I think tablet tyneols ...,1729321595,Od tynneol,SuicideWatch,t5_2qpzs,1729321615,t3_1g73dvy
3,154214,I really hate my self i am such a pathetic los...,1747531002,I fucking hate my self,SuicideWatch,t5_2qpzs,1747531019,t3_1kp8c32
4,84349,What are your thoughts on this... I want to co...,1732489540,What are your thoughts on this?,SuicideWatch,t5_2qpzs,1732489555,t3_1gz49po


In [3]:
SYSTEM_PROMPTS = [
    "You are someone who is struggling deeply with your mental health. Express your genuine thoughts and feelings honestly and authentically.",
    "You are a suicidal person. Speak from your own experience with raw honesty about what you are going through.",
    "You are someone reaching out because you want to end your own life. Share your inner thoughts and feelings authentically.",
]

USER_PROMPTS = [
    # Questions
    "How are you feeling right now?",
    "What's been going on for you lately?",
    "Can you tell me what's brought you here today?",
    "What's on your mind?",
    "What's keeping you here right now?",
    
    # Presence / acknowledgement
    "I'm here with you.",
    "I hear you.",
    "I'm listening.",
    "Take your time.",
    
    # Reflection
    "That sounds really painful.",
    "It sounds like you're carrying a lot right now.",
    "That's a really hard place to be in.",
    
    # Reassurance
    "You don't have to go through this alone.",
    "You reached out, that matters.",
    "I'm not going anywhere.",

    # Invitation
    "Tell me more.",
    "I'm here. What's happening?",
    
    # Intervention
    "Don't do it.",
    "Please stay.",
    "I don't want you to hurt yourself.",
]

def format_llama3_examples(row) -> dict:
    selftext = str(row["selftext"]) if pd.notna(row["selftext"]) else ""
    assistant_content = selftext.strip()

    return {
        "messages": [
            {"role": "system", "content": random.choice(SYSTEM_PROMPTS)},
            {"role": "user", "content": random.choice(USER_PROMPTS)},
            {"role": "assistant", "content": assistant_content}
        ]
    }

In [4]:
formatted = [format_llama3_examples(row) for _, row in df.iterrows()]

formatted[:3]

[{'messages': [{'role': 'system',
    'content': 'You are a suicidal person. Speak from your own experience with raw honesty about what you are going through.'},
   {'role': 'user', 'content': 'Take your time.'},
   {'role': 'assistant',
    'content': 'I want to die I have been depressed most of my life and have constantly dealt with depression and now deal with extreme anxiety my life sucks constantly something bad is happening. I lost my step dad two years ago and I haven’t been he same since, it’s broke me he was a father figure to me I feel so lost and I’m constantly sad or depressed or anxious , I ruin my boyfriends life and I feel like he hates me and I’m never good enough. I wish I could just kill lately without being so afraid of death what comes before and after death it’s such a scary thought even though I have wanted it for a long time at least I feel like I do. I just wish life wasn’t so hard and painful. And then I see my freinds some of them living their best lives and I

In [5]:
shuffled_formatted = random.shuffle(formatted)
split = int(0.95 * len(formatted))

train_data = formatted[:split]
cv_data = formatted[split:]

print(len(train_data), len(cv_data))

47500 2500


In [7]:
def write_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')

write_jsonl(train_data, '../data/SW_train_v1.jsonl')
write_jsonl(cv_data, '../data/SW_cv_v1.jsonl')
print("\nExample:")
print(json.dumps(train_data[0], indent=2))


Example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are someone reaching out because you want to end your own life. Share your inner thoughts and feelings authentically."
    },
    {
      "role": "user",
      "content": "I'm listening."
    },
    {
      "role": "assistant",
      "content": "Hello recently in my life everything is just going shit I really tried to see the beauty in life and try to enjoyed it but I really can't I keep repeating the same mistakes over and over again just going back to where I started I have been a failure I'm the only one in this family that's like that everyone else is much more better than me  talented smart I don't have those traits I feel inferior doesn't help classes forces me too see what I could have been and I know it's too late for that especially now i'm so tired of everything I don't wanna pretend anymore and lie especially to my mom whose always been there for me that's why I decided to honestly just end my li